## Analisis Sentimen Aplikasi dan Gerai Kopi Kenangan
### Data Preprocessing (Functional Approach)

In [2]:
!pip install Sastrawi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.7/209.7 kB 4.5 MB/s eta 0:00:00


In [23]:
import os
import re
import string
import pandas as pd
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt', quiet=True)

try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords', quiet=True)

try:
    nltk.data.find('tokenizers/punkt_tab')
except LookupError:
    nltk.download('punkt_tab', quiet=True)

### Kamus Slang (Slang Dictionary)

In [4]:
# Kamus slang untuk normalisasi bahasa gaul / singkatan ulasan
slang_dict = {
    "apk": "aplikasi",
    "bgus": "bagus",
    "tdk": "tidak",
    "g": "tidak",
    "gk": "tidak",
    "ga": "tidak",
    "bgt": "banget",
    "krn": "karena",
    "app": "aplikasi",
    "sy": "saya",
    "yg": "yang",
    "dgn": "dengan",
    "utk": "untuk",
    "klo": "kalau",
    "tp": "tapi",
    "dpt": "dapat",
    "sm": "sama",
    "rame": "ramai",
    "cozy": "nyaman",
    "afford": "terjangkau",
    "affordable": "terjangkau",
    "bgtu": "begitu",
    "cmn": "cuma",
    "jg": "juga",
    "karna": "karena",
    "kemaren": "kemarin",
    "mulu": "melulu",
    "nggak": "tidak",
    "dapet": "dapat",
    "bener": "benar",
    "beneran": "benar-benar",
    "wfc": "kerja dari kafe",
    "nongki": "nongkrong",
    "kopken": "kopi kenangan",
    "enakkk": "enak",
    "enakk": "enak",
    "enak-enak": "enak",
    "mantappp": "mantap",
    "mantap": "mantap",
    "josss": "bagus",
    "st*rb*cks": "starbucks",
    "starbuck": "starbucks",
    "lbh": "lebih",
    "cm": "cuma",
    "kl": "kalau",
    "hr": "hari",
    "mgkn": "mungkin",
    "sayanya": "saya",
    "baperista": "barista",
    "kokas": "kota kasablanka",
    "sency": "senayan city",
    "blok m": "blok m",
    "mrt": "mrt",
    "laper": "lapar",
    "nyoba": "mencoba",
    "nyobain": "mencoba",
    "kemis": "kamis",
    "jum'at": "jumat",
    "weekend": "akhir pekan",
    "weekdays": "hari kerja",
    "slow": "lambat",
    "lelet": "lambat",
    "gercep": "cepat",
}

print(f"Kamus slang didefinisikan dengan {len(slang_dict)} kata.")

Kamus slang didefinisikan dengan 63 kata.


### Fungsi Pembersih Teks & Preprocessing

In [5]:
# Inisialisasi list stopword gabungan
stopword_indo = set(stopwords.words('indonesian'))
stopword_eng = set(stopwords.words('english'))
stopword_indo.update(stopword_eng)

# custom stopword dari contoh.ipynb
custom_stopwords = ['sih', 'nya', 'kok', 'yah', 'dong', 'kan', 'iya', 'yaa', 'gak', 'na', 'ku', 'di', 'ga', 'ya', 'gaa', 'loh', 'kah', 'woi', 'woii', 'woy']
stopword_indo.update(custom_stopwords)

# Inisialisasi Sastrawi Stemmer
print("Membuat Sastrawi Stemmer...")
factory = StemmerFactory()
stemmer = factory.create_stemmer()

# Global dictionary cache untuk mempercepat stemming Sastrawi
stem_cache = {}

Membuat Sastrawi Stemmer...


In [6]:
def get_cached_stem(word, stemmer_obj):
    """Fungsi stemming dengan cache agar proses berjalan 10-20x lebih cepat"""
    if not word:
        return ""
    if word not in stem_cache:
        stem_cache[word] = stemmer_obj.stem(word)
    return stem_cache[word]

In [7]:
def clean_noise(text):
    """Membersihkan emoji, URL, mention, tanda baca, dan teks '... Lainnya'"""
    if not isinstance(text, str):
        return ""

    # Hilangkan akhiran ulasan '... Lainnya'
    text = re.sub(r'\.{2,}\s*Lainnya', '', text)
    # Hilangkan mention @username
    text = re.sub(r'@[A-Za-z0-9_]+', '', text)
    # Hilangkan hashtag #tag
    text = re.sub(r'#[A-Za-z0-9_]+', '', text)
    # Hilangkan RT
    text = re.sub(r'\bRT\b', '', text)
    # Hilangkan URL
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    # Hilangkan angka
    text = re.sub(r'\d+', '', text)
    # Ganti baris baru dengan spasi
    text = text.replace('\n', ' ')
    # Hilangkan karakter non-ASCII (emoji)
    text = text.encode('ascii', 'ignore').decode('ascii')
    # Hilangkan tanda baca
    text = text.translate(str.maketrans('', '', string.punctuation))
    # Satukan spasi berlebih
    text = ' '.join(text.split())
    # Case folding ke huruf kecil
    return text.lower()

In [25]:
def preprocess_review(text):
    # pipeline preprocessing
    try:
        # Cleaning Noise & Case Folding
        cleaned = clean_noise(text)
        if not cleaned:
            return ""
        # Tokenisasi
        tokens = word_tokenize(cleaned)
        # Normalisasi Slang
        normalized = [slang_dict.get(t, t) for t in tokens]
        # Filter Stopwords
        filtered = [t for t in normalized if t not in stopword_indo]
        # Stemming dengan Cache
        stemmed = [get_cached_stem(t, stemmer) for t in filtered if t]
        # Gabungkan
        return ' '.join(stemmed)
    except Exception as e:
        print(f"[ERROR] Failed to process review: '{text[:100]}...' - Error: {e}")
        return ""

### Load Dataset & Jalankan Pipeline

In [9]:
dataset_path = 'cleaned_data.csv'
if os.path.exists(dataset_path):
    df = pd.read_csv(dataset_path)
    print(f"Dataset berhasil dimuat! Jumlah data: {len(df)} baris.")
    print(df.head(2))
else:
    print(f"[ERROR] File {dataset_path} tidak ditemukan di folder root!")

Dataset berhasil dimuat! Jumlah data: 2938 baris.
     nama_pengulas                                             ulasan  \
0            Immia  Sekarang udah jadi Kenangan boulangerie, semak...   
1  Anton Kristanto  Outlet kopi kenangan versi premium... The best...   

      rating  
0  5 bintang  
1  5 bintang  


In [26]:
print("Menjalankan text preprocessing pada seluruh ulasan...")
# Jalankan pipeline pembersihan
df['text_clean'] = df['ulasan'].apply(preprocess_review)
print("Pembersihan selesai!")

Menjalankan text preprocessing pada seluruh ulasan...
Pembersihan selesai!


In [27]:
empty_cleaned_reviews = df[df['text_clean'] == ''].shape[0]
total_reviews = df.shape[0]
print(f"Jumlah ulasan yang menjadi kosong setelah preprocessing: {empty_cleaned_reviews} dari {total_reviews}")
if empty_cleaned_reviews == total_reviews:
    print("Semua ulasan menjadi kosong setelah preprocessing. Ini adalah masalah utama.")

Jumlah ulasan yang menjadi kosong setelah preprocessing: 11 dari 2938


### Inspecting Preprocessing Results

In [28]:
print("Original vs. Preprocessed Reviews (First 10 samples):")
for i in range(10):
    original_review = df['ulasan'].iloc[i]
    preprocessed_review = df['text_clean'].iloc[i]
    print(f"\nReview #{i+1}:")
    print(f"  Original: {original_review}")
    print(f"  Cleaned:  {preprocessed_review}")

Original vs. Preprocessed Reviews (First 10 samples):

Review #1:
  Original: Sekarang udah jadi Kenangan boulangerie, semakin keren tempatnya dan lebih banyak pilihan rotinya. Ada mesin untuk self order juga. Ada banyak colokan juga buat yang mau sambil wfc. Kopi nya juga makin banyak pilihan walau kopi kenangan mantan masih jadi favorit 😄 … Lainnya
  Cleaned:  udah kenang boulangerie keren tempat pilih roti mesin self order colok kerja dari kafe kopi pilih kopi kenang mantan favorit

Review #2:
  Original: Outlet kopi kenangan versi premium... The best banget pelayanan dan suasananya. Sangat mewah
Jenis pe…
Lainnya
  Cleaned:  outlet kopi kenang versi premium best banget layan suasana mewah jenis pe

Review #3:
  Original: kalo makan siang sering beli kopken walaupun antrian panjang banget tapi pelayanannya tetep gercep salut banget👍kopinya juga enakkk banget😋🫰 … Lainnya
  Cleaned:  kalo makan siang beli kopi kenang antri banget layan tetep cepat salut bangetkopinya enak banget

Revi

### Pelabelan Sentimen berdasarkan Rating & Filtering

In [31]:
# Hitung jumlah kata setelah preprocessing
df['word_count'] = df['text_clean'].apply(lambda x: len(str(x).split()))

In [32]:
# Filter baris dengan kata kurang dari 1 (yaitu, hanya biarkan yang ada isinya)
print(f"Jumlah baris sebelum difilter: {len(df)}")
df_filtered = df[df['word_count'] >= 1].copy()
print(f"Jumlah baris setelah difilter (minimal 1 kata): {len(df_filtered)}")

Jumlah baris sebelum difilter: 2938
Jumlah baris setelah difilter (minimal 1 kata): 2927


In [33]:
# Konversi Rating ulasan (Contoh: "5 bintang" -> 5) menjadi kelas sentimen
def map_rating_to_sentiment(rating_str):
    if not isinstance(rating_str, str):
        return 'neutral'
    match = re.search(r'\d', rating_str)
    if match:
        stars = int(match.group())
        if stars >= 4:
            return 'positive'
        elif stars <= 2:
            return 'negative'
    return 'neutral'

In [34]:
if 'rating' in df_filtered.columns:
    df_filtered['sentiment'] = df_filtered['rating'].apply(map_rating_to_sentiment)
    print("\nDistribusi Kelas Sentimen:")
    print(df_filtered['sentiment'].value_counts())
else:
    print("[WARNING] Kolom rating tidak ditemukan, ulasan tidak dapat dilabeli otomatis.")


Distribusi Kelas Sentimen:
sentiment
positive    2301
negative     498
neutral      128
Name: count, dtype: int64


### Hasil Preprocessing

In [35]:
# Simpan hasil preprocessing
output_path = 'cleaned_reviews.csv'
df_filtered.to_csv(output_path, index=False)
print(f"Proses preprocessing berhasil diselesaikan!")
print(f"Data bersih disimpan ke: {output_path}")

Proses preprocessing berhasil diselesaikan!
Data bersih disimpan ke: cleaned_reviews.csv
